In [9]:
pip install scikit_posthocs

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Import dependencies

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [6]:


data_add = "../data/marketing_data_cleaned.xlsx"

df = pd.read_excel(data_add)

# ROAS = Revenue / Spend
df["ROAS"] = np.where(
    df["Spend"] != 0,
    df["Revenue"] / df["Spend"],
    np.nan
)

# CTR = Clicks / Impressions
df["CTR"] = np.where(
    df["Impressions"] != 0,
    df["Clicks"] / df["Impressions"],
    np.nan
)

# CVR = Purchases / Clicks
df["CVR"] = np.where(
    df["Clicks"] != 0,
    df["Purchases"] / df["Clicks"],
    np.nan
)

# CPC = Spend / Clicks
df["CPC"] = np.where(
    df["Clicks"] != 0,
    df["Spend"] / df["Clicks"],
    np.nan
)

# (Optional) Round for readability
df["ROAS"] = df["ROAS"].round(4)
df["CTR"] = df["CTR"].round(4)
df["CVR"] = df["CVR"].round(4)
df["CPC"] = df["CPC"].round(4)

df.head()


,Date,Platform,Campaign,Region,Spend,CPM,Impressions,Frequency,Clicks,Purchases,...,Video_Completion_Rate,Customer_LTV,Is_Competitive_Event,CPM_calc,Calculated_CPM,campaign_id,ROAS,CTR,CVR,CPC
0,2024-01-01,FB,FB_Athletes_Video_Protein_001,West,1187.29,18.0,65960,2.75,527,15,...,0.486,772.8,True,18.000152,18.000152,1,1.0248,0.0080,0.0285,2.2529
1,2024-01-01,FB,FB_Athletes_Video_Protein_001,South,1081.89,18.0,60105,2.51,512,12,...,0.486,772.8,True,18.000000,18.000000,1,0.8024,0.0085,0.0234,2.1131
2,2024-01-01,FB,FB_Athletes_Video_Protein_001,Northeast,648.47,18.0,36025,2.52,344,9,...,0.486,772.8,True,18.000555,18.000555,1,1.0713,0.0095,0.0262,1.8851
3,2024-01-01,FB,FB_Athletes_Video_Protein_001,Midwest,463.38,18.0,25743,2.49,272,6,...,0.486,772.8,True,18.000233,18.000233,1,1.0416,0.0106,0.0221,1.7036
4,2024-01-01,FB,FB_Athletes_Image_Preworkout_002,West,800.59,18.0,44477,3.10,418,14,...,NaN,538.2,True,18.000090,18.000090,2,1.0645,0.0094,0.0335,1.9153


In [13]:
df[df['Clicks'] == 0].head(20)

,Date,Platform,Campaign,Region,Spend,CPM,Impressions,Frequency,Clicks,Purchases,...,Video_Completion_Rate,Customer_LTV,Is_Competitive_Event,CPM_calc,Calculated_CPM,campaign_id,ROAS,CTR,CVR,CPC
85,2024-01-03,Google,Google_Athletes_Search_Protein_001,Northeast,1303.24,15.03,86709,2.61,0,41,...,NaN,940.80,False,15.030043,15.030043,1,2.7571,0.0,NaN,NaN
91,2024-01-03,Google,Google_FitnessEnth_Display_Preworkout_001,West,469.88,15.03,31262,2.57,0,13,...,NaN,546.00,False,15.030388,15.030388,1,1.5221,0.0,NaN,NaN
93,2024-01-03,Google,Google_FitnessEnth_Display_Preworkout_001,Northeast,921.64,15.03,61320,3.77,0,12,...,NaN,546.00,False,15.030007,15.030007,1,0.7501,0.0,NaN,NaN
121,2024-01-04,Google,Google_Athletes_Search_Protein_001,Northeast,642.88,15.04,42730,3.42,0,16,...,NaN,940.80,False,15.045167,15.045167,1,2.1599,0.0,NaN,NaN
159,2024-01-05,Google,Google_WeightLoss_Search_Diet_001,West,1508.80,15.06,100186,3.79,0,47,...,NaN,393.12,False,15.059988,15.059988,1,1.8598,0.0,NaN,NaN
161,2024-01-05,Google,Google_WeightLoss_Search_Diet_001,Northeast,1576.44,15.06,104677,2.88,0,70,...,NaN,393.12,False,15.060042,15.060042,1,2.8131,0.0,NaN,NaN
200,2024-01-06,Google,Google_FitnessEnth_Display_Preworkout_001,Midwest,436.89,18.09,24150,2.75,0,5,...,NaN,546.00,False,18.090683,18.090683,1,0.6409,0.0,NaN,NaN
227,2024-01-07,Google,Google_Athletes_Search_Protein_001,Northeast,1076.51,18.11,59449,3.86,0,34,...,NaN,940.80,False,18.108126,18.108126,1,2.3948,0.0,NaN,NaN
297,2024-01-09,Google,Google_Athletes_Search_Protein_001,Northeast,969.32,15.12,64108,2.73,0,33,...,NaN,940.80,False,15.120110,15.120110,1,2.6236,0.0,NaN,NaN
304,2024-01-09,Google,Google_FitnessEnth_Display_Preworkout_001,South,516.06,15.12,34130,3.16,0,9,...,NaN,546.00,False,15.120422,15.120422,1,0.9701,0.0,NaN,NaN


In [14]:
import pandas as pd
import numpy as np

from scipy.stats import (
    shapiro,
    levene,
    f_oneway,
    kruskal
)

from statsmodels.stats.multicomp import pairwise_tukeyhsd
import scikit_posthocs as sp


def compare_groups(df, group_col, metrics=None, alpha=0.05):
    """
    Compare multiple metrics across a categorical variable.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset.

    group_col : str
        Categorical column to compare
        (e.g. 'Platform', 'Region', 'Creative_Type').

    metrics : list[str], optional
        Metrics to analyze.
        Default = ["ROAS", "CTR", "CVR", "CPC"]

    alpha : float
        Significance level.
    """

    if metrics is None:
        metrics = ["ROAS", "CTR", "CVR", "CPC"]

    groups_names = df[group_col].dropna().unique()

    for metric in metrics:

        print("=" * 80)
        print(f"{metric} by {group_col}")
        print("=" * 80)

        # -------------------------------------------------------------
        # Descriptive statistics
        # -------------------------------------------------------------

        summary = (
            df.groupby(group_col)[metric]
            .agg(["mean", "median", "std", "count"])
            .round(4)
        )

        print("\nDescriptive Statistics")
        print(summary)

        # -------------------------------------------------------------
        # Gather groups
        # -------------------------------------------------------------

        groups = [
            df[df[group_col] == g][metric].dropna()
            for g in groups_names
        ]

        # -------------------------------------------------------------
        # Shapiro-Wilk
        # -------------------------------------------------------------

        normal = True

        print("\nShapiro-Wilk Test")

        for name, values in zip(groups_names, groups):

            if len(values) >= 3:

                _, pvalue = shapiro(values)

                print(f"{name:<20} p = {pvalue:.4f}")

                if pvalue < alpha:
                    normal = False

        # -------------------------------------------------------------
        # Levene
        # -------------------------------------------------------------

        _, lev_p = levene(*groups)

        equal_var = lev_p >= alpha

        print(f"\nLevene Test p = {lev_p:.4f}")

        # -------------------------------------------------------------
        # Parametric / Non-parametric
        # -------------------------------------------------------------

        if normal and equal_var:

            print("\nUsing One-Way ANOVA")

            stat, p = f_oneway(*groups)

            print(f"F = {stat:.4f}")
            print(f"p = {p:.6f}")

            if p < alpha:

                print("\nSignificant differences detected.")

                tukey = pairwise_tukeyhsd(
                    endog=df[metric],
                    groups=df[group_col],
                    alpha=alpha
                )

                print("\nTukey HSD")
                print(tukey)

            else:

                print("\nNo statistically significant differences.")

        else:

            print("\nUsing Kruskal-Wallis Test")

            stat, p = kruskal(*groups)

            print(f"H = {stat:.4f}")
            print(f"p = {p:.6f}")

            if p < alpha:

                print("\nSignificant differences detected.")

                dunn = sp.posthoc_dunn(
                    df,
                    val_col=metric,
                    group_col=group_col,
                    p_adjust="bonferroni"
                )

                print("\nDunn Post-hoc Test")
                print(dunn.round(4))

            else:

                print("\nNo statistically significant differences.")

        # -------------------------------------------------------------
        # Business Summary
        # -------------------------------------------------------------

        best = summary["mean"].idxmax()
        worst = summary["mean"].idxmin()

        print("\nBusiness Summary")
        print("-" * 28)

        print(
            f"The highest average {metric} was achieved by "
            f"{best} ({summary.loc[best,'mean']:.4f})."
        )

        print(
            f"The lowest average {metric} was observed for "
            f"{worst} ({summary.loc[worst,'mean']:.4f})."
        )

        if p < alpha:

            print(
                f"Differences between {group_col.lower()}s "
                f"are statistically significant (p < {alpha})."
            )

        else:

            print(
                f"No statistically significant differences were found "
                f"between {group_col.lower()}s (p ≥ {alpha})."
            )

        print("\n\n")

In [15]:
compare_groups(df, "Platform")

ROAS by Platform

Descriptive Statistics
            mean  median     std  count
Platform                               
FB        0.8162  0.7499  0.4225   1054
Google    1.1057  0.9456  0.6495   1053
TT        0.9602  0.8625  0.4950   1066

Shapiro-Wilk Test
FB                   p = 0.0000
Google               p = 0.0000
TT                   p = 0.0000

Levene Test p = 0.0000

Using Kruskal-Wallis Test
H = 106.6989
p = 0.000000

Significant differences detected.

Dunn Post-hoc Test
         FB  Google      TT
FB      1.0  0.0000  0.0000
Google  0.0  1.0000  0.0002
TT      0.0  0.0002  1.0000

Business Summary
----------------------------
The highest average ROAS was achieved by Google (1.1057).
The lowest average ROAS was observed for FB (0.8162).
Differences between platforms are statistically significant (p < 0.05).



CTR by Platform

Descriptive Statistics
            mean  median     std  count
Platform                               
FB        0.0057  0.0056  0.0021   1054
Google

In [16]:
compare_groups(df, "Region")

ROAS by Region

Descriptive Statistics
             mean  median     std  count
Region                                  
Midwest    0.8337  0.6951  0.4940    760
Northeast  1.0330  0.8678  0.5960    803
South      0.9605  0.8535  0.4868    803
West       1.0083  0.8915  0.5662    807

Shapiro-Wilk Test
West                 p = 0.0000
South                p = 0.0000
Northeast            p = 0.0000
Midwest              p = 0.0000

Levene Test p = 0.0000

Using Kruskal-Wallis Test
H = 69.2321
p = 0.000000

Significant differences detected.

Dunn Post-hoc Test
           Midwest  Northeast  South  West
Midwest        1.0        0.0    0.0   0.0
Northeast      0.0        1.0    1.0   1.0
South          0.0        1.0    1.0   1.0
West           0.0        1.0    1.0   1.0

Business Summary
----------------------------
The highest average ROAS was achieved by Northeast (1.0330).
The lowest average ROAS was observed for Midwest (0.8337).
Differences between regions are statistically significa

In [17]:
compare_groups(df, "Product_Category")

ROAS by Product_Category

Descriptive Statistics
                    mean  median     std  count
Product_Category                               
Diet              1.0578  0.9574  0.5374    711
Preworkout        0.6184  0.5716  0.2522   1053
Protein           1.3059  1.1981  0.5542   1054
WeightLoss        0.7560  0.6572  0.4480    355

Shapiro-Wilk Test
Protein              p = 0.0000
Preworkout           p = 0.0000
WeightLoss           p = 0.0000
Diet                 p = 0.0000

Levene Test p = 0.0000

Using Kruskal-Wallis Test
H = 1089.1533
p = 0.000000

Significant differences detected.

Dunn Post-hoc Test
            Diet  Preworkout  Protein  WeightLoss
Diet         1.0         0.0      0.0         0.0
Preworkout   0.0         1.0      0.0         0.0
Protein      0.0         0.0      1.0         0.0
WeightLoss   0.0         0.0      0.0         1.0

Business Summary
----------------------------
The highest average ROAS was achieved by Protein (1.3059).
The lowest average ROAS was

In [18]:
compare_groups(df, "Creative_Type")

ROAS by Creative_Type

Descriptive Statistics
                 mean  median     std  count
Creative_Type                               
Carousel       0.7560  0.6572  0.4480    355
Display        0.6626  0.6194  0.2685    348
Image          0.6176  0.5656  0.2749    349
Search         1.3244  1.2455  0.6708    705
Video          0.9886  0.9058  0.4730   1416

Shapiro-Wilk Test
Video                p = 0.0000
Image                p = 0.0000
Carousel             p = 0.0000
Search               p = 0.0000
Display              p = 0.0000

Levene Test p = 0.0000

Using Kruskal-Wallis Test
H = 606.8404
p = 0.000000

Significant differences detected.

Dunn Post-hoc Test
          Carousel  Display   Image  Search  Video
Carousel    1.0000   0.2527  0.0008     0.0    0.0
Display     0.2527   1.0000  0.8940     0.0    0.0
Image       0.0008   0.8940  1.0000     0.0    0.0
Search      0.0000   0.0000  0.0000     1.0    0.0
Video       0.0000   0.0000  0.0000     0.0    1.0

Business Summary
----

In [19]:
compare_groups(df, "Target_Audience")

ROAS by Target_Audience

Descriptive Statistics
                   mean  median     std  count
Target_Audience                               
Athletes         1.1347  1.0448  0.5814   1403
FitnessEnth      0.6648  0.6022  0.3313   1059
WeightLoss       1.0578  0.9574  0.5374    711

Shapiro-Wilk Test
Athletes             p = 0.0000
FitnessEnth          p = 0.0000
WeightLoss           p = 0.0000

Levene Test p = 0.0000

Using Kruskal-Wallis Test
H = 566.6095
p = 0.000000

Significant differences detected.

Dunn Post-hoc Test
             Athletes  FitnessEnth  WeightLoss
Athletes       1.0000          0.0      0.0122
FitnessEnth    0.0000          1.0      0.0000
WeightLoss     0.0122          0.0      1.0000

Business Summary
----------------------------
The highest average ROAS was achieved by Athletes (1.1347).
The lowest average ROAS was observed for FitnessEnth (0.6648).
Differences between target_audiences are statistically significant (p < 0.05).



CTR by Target_Audience

Descri